# Notebook 3: Graph Neural Network (GraphSAGE)
Run Notebooks 1 and 2 first.

Models the transaction ecosystem as a graph:
- Nodes = unique accounts (origin + destination)
- Edges = transactions between accounts
- Task = node classification (is this account involved in fraud?)

Architecture: GraphSAGE (inductive — handles new accounts at inference).
Zakaria et al. (2025) showed GNN+anomaly hybrid achieves ROC-AUC 0.973.

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    classification_report
)
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import warnings, json
warnings.filterwarnings('ignore')

# PyTorch Geometric
try:
    from torch_geometric.data import Data
    from torch_geometric.nn import SAGEConv
    from torch_geometric.utils import to_undirected
    print('PyTorch Geometric loaded')
except ImportError:
    print('Installing PyTorch Geometric...')
    import subprocess
    subprocess.run(['pip', 'install', 'torch-geometric', '--quiet'])
    from torch_geometric.data import Data
    from torch_geometric.nn import SAGEConv
    from torch_geometric.utils import to_undirected

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## 1. Load and Sample Data

Full PaySim has 6.3M+ unique accounts — impractical for a single-GPU GNN.
We sample strategically: ALL fraud transactions + 15x that many normals.
This gives a representative, manageable graph while maintaining signal.

In [ ]:
train_df = pd.read_csv('outputs/train_df.csv')
test_df  = pd.read_csv('outputs/test_df.csv')

# Work on TRANSFER + CASH_OUT only
train_df = train_df[train_df['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()
test_df  = test_df[test_df['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()

# Sample: all fraud + 15x normal
def stratified_sample(df, fraud_col='isFraud', normal_ratio=15):
    fraud  = df[df[fraud_col] == 1]
    normal = df[df[fraud_col] == 0].sample(
        n=min(len(fraud) * normal_ratio, len(df[df[fraud_col]==0])),
        random_state=42
    )
    return pd.concat([fraud, normal]).sort_values('step').reset_index(drop=True)

train_sample = stratified_sample(train_df, normal_ratio=15)
test_sample  = test_df  # use full test for fair evaluation

print(f'Train sample: {len(train_sample):,} | Fraud: {train_sample["isFraud"].sum()}')
print(f'Test:         {len(test_sample):,}  | Fraud: {test_sample["isFraud"].sum()}')

## 2. Build Account Graph

In [ ]:
def build_graph(df):
    """
    Build a transaction graph from a DataFrame.

    Nodes: unique accounts (nameOrig + nameDest)
    Edges: transactions (directed: orig → dest)
    Node features: aggregated stats per account
    Node label: 1 if account appears in any fraud transaction
    """
    # Collect all unique accounts
    all_accounts = pd.unique(df[['nameOrig', 'nameDest']].values.ravel())
    account_to_idx = {acc: i for i, acc in enumerate(all_accounts)}
    n_nodes = len(all_accounts)

    # Node features: aggregate transaction stats per account
    def get_node_features(df, account_col, prefix):
        grp = df.groupby(account_col).agg(
            txn_count   =('amount', 'count'),
            mean_amount =('amount', 'mean'),
            total_amount=('amount', 'sum'),
            max_amount  =('amount', 'max'),
            std_amount  =('amount', 'std'),
            mean_balance=('oldbalanceOrg', 'mean') if 'oldbalanceOrg' in df else ('amount', 'mean')
        ).fillna(0)
        grp.columns = [f'{prefix}_{c}' for c in grp.columns]
        return grp

    orig_feats = get_node_features(df, 'nameOrig', 'orig')
    dest_feats = get_node_features(df, 'nameDest', 'dest')

    # Build node feature matrix (zeros for missing accounts)
    n_orig_feats = orig_feats.shape[1]
    n_dest_feats = dest_feats.shape[1]
    node_feats = np.zeros((n_nodes, n_orig_feats + n_dest_feats), dtype=np.float32)

    for acc, idx in account_to_idx.items():
        if acc in orig_feats.index:
            node_feats[idx, :n_orig_feats] = orig_feats.loc[acc].values
        if acc in dest_feats.index:
            node_feats[idx, n_orig_feats:] = dest_feats.loc[acc].values

    # Scale
    scaler = StandardScaler()
    node_feats = scaler.fit_transform(node_feats).astype(np.float32)

    # Node labels: fraud if involved in any fraud transaction
    fraud_orig = set(df[df['isFraud']==1]['nameOrig'])
    fraud_dest = set(df[df['isFraud']==1]['nameDest'])
    fraud_accs = fraud_orig | fraud_dest
    node_labels = np.array([1 if acc in fraud_accs else 0
                            for acc in all_accounts], dtype=np.int64)

    # Edge index
    src = [account_to_idx[o] for o in df['nameOrig']]
    dst = [account_to_idx[d] for d in df['nameDest']]
    edge_index = torch.tensor([src, dst], dtype=torch.long)
    edge_index = to_undirected(edge_index)  # undirected graph

    data = Data(
        x=torch.FloatTensor(node_feats),
        edge_index=edge_index,
        y=torch.LongTensor(node_labels)
    )

    print(f'Graph: {n_nodes:,} nodes, {edge_index.shape[1]:,} edges')
    print(f'Fraud nodes: {node_labels.sum():,} ({node_labels.mean()*100:.2f}%)')
    print(f'Node features: {node_feats.shape[1]}')
    return data, account_to_idx, scaler

print('Building training graph...')
train_data, train_a2i, node_scaler = build_graph(train_sample)

# Train/val split at node level (random)
n_nodes = train_data.num_nodes
perm    = torch.randperm(n_nodes)
val_cut = int(n_nodes * 0.85)
train_data.train_mask = torch.zeros(n_nodes, dtype=torch.bool)
train_data.val_mask   = torch.zeros(n_nodes, dtype=torch.bool)
train_data.train_mask[perm[:val_cut]] = True
train_data.val_mask[perm[val_cut:]]   = True

train_data = train_data.to(DEVICE)

## 3. GraphSAGE Model

In [ ]:
class GraphSAGEFraud(nn.Module):
    """
    GraphSAGE for fraud node classification.

    Why GraphSAGE over GAT?
    - Inductive: handles new accounts at inference time without retraining
    - More efficient on sparse graphs (PaySim is relatively sparse)
    - Demonstrated in Zakaria et al. (2025) for financial fraud

    3-layer design: captures 3-hop neighbourhoods (account → merchant → ring)
    """
    def __init__(self, in_channels, hidden=128, out_classes=2, dropout=0.3):
        super().__init__()
        self.conv1   = SAGEConv(in_channels, hidden)
        self.conv2   = SAGEConv(hidden, hidden)
        self.conv3   = SAGEConv(hidden, hidden // 2)
        self.dropout = nn.Dropout(dropout)
        self.bn1     = nn.BatchNorm1d(hidden)
        self.bn2     = nn.BatchNorm1d(hidden)
        self.classifier = nn.Linear(hidden // 2, out_classes)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.dropout(x)

        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.dropout(x)

        x = self.conv3(x, edge_index)
        x = F.relu(x)

        return self.classifier(x)  # (n_nodes, 2)

    def get_embeddings(self, x, edge_index):
        """Returns node embeddings before classification head."""
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = self.dropout(x)
        x = F.relu(self.bn2(self.conv2(x, edge_index)))
        x = self.dropout(x)
        x = F.relu(self.conv3(x, edge_index))
        return x


n_features = train_data.x.shape[1]
gnn_model = GraphSAGEFraud(in_channels=n_features, hidden=128, dropout=0.3).to(DEVICE)

# Class weights for imbalanced training
fraud_count  = train_data.y[train_data.train_mask].sum().item()
total_count  = train_data.train_mask.sum().item()
class_weight = torch.tensor([
    1.0,
    (total_count - fraud_count) / (fraud_count + 1)
], dtype=torch.float).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weight)
optimizer = torch.optim.Adam(gnn_model.parameters(), lr=5e-4, weight_decay=1e-4)

params = sum(p.numel() for p in gnn_model.parameters() if p.requires_grad)
print(f'GraphSAGE parameters: {params:,}')
print(f'Class weight for fraud: {class_weight[1].item():.1f}x')

## 4. Training Loop

In [ ]:
EPOCHS   = 100
best_f1  = 0
tr_losses, val_f1s = [], []

print(f'Training GraphSAGE for {EPOCHS} epochs...')
print(f'{"Epoch":>6} {"Train Loss":>12} {"Val F1":>8} {"Val Rec":>8} {"Val Prec":>10}')
print('-' * 55)

for epoch in range(1, EPOCHS + 1):
    # Train
    gnn_model.train()
    optimizer.zero_grad()
    out  = gnn_model(train_data.x, train_data.edge_index)
    loss = criterion(out[train_data.train_mask], train_data.y[train_data.train_mask])
    loss.backward()
    optimizer.step()

    # Validate
    gnn_model.eval()
    with torch.no_grad():
        val_out   = gnn_model(train_data.x, train_data.edge_index)
        val_preds = val_out[train_data.val_mask].argmax(dim=1).cpu().numpy()
        val_true  = train_data.y[train_data.val_mask].cpu().numpy()

    vf1  = f1_score(val_true, val_preds, zero_division=0)
    vrec = recall_score(val_true, val_preds, zero_division=0)
    vprc = precision_score(val_true, val_preds, zero_division=0)

    tr_losses.append(loss.item())
    val_f1s.append(vf1)

    if vf1 > best_f1:
        best_f1 = vf1
        torch.save(gnn_model.state_dict(), 'outputs/best_gnn_model.pt')

    if epoch % 10 == 0 or epoch <= 5:
        print(f'{epoch:>6} {loss.item():>12.5f} {vf1:>8.4f} {vrec:>8.4f} {vprc:>10.4f}')

print(f'\nBest Val F1: {best_f1:.4f}')

## 5. Build Test Graph & Evaluate

In [ ]:
# Load best model
gnn_model.load_state_dict(torch.load('outputs/best_gnn_model.pt', map_location=DEVICE))
gnn_model.eval()

# Build test graph
print('Building test graph...')
test_data, test_a2i, _ = build_graph(test_sample)
test_data = test_data.to(DEVICE)

# For test: we need a GNN model that can handle new graph
# Re-initialise with same architecture but train fresh or use transfer
# Practical approach: use train graph structure + node features for unseen nodes
# Here we evaluate on the test graph directly (transductive on test subgraph)

# If test graph has different feature size, rebuild model
test_n_features = test_data.x.shape[1]
if test_n_features != n_features:
    print(f'Feature mismatch: train={n_features}, test={test_n_features}. Padding...')
    pad = torch.zeros(test_data.x.shape[0], abs(test_n_features - n_features)).to(DEVICE)
    test_data.x = torch.cat([test_data.x, pad], dim=1) if test_n_features < n_features \
                  else test_data.x[:, :n_features]

with torch.no_grad():
    test_out    = gnn_model(test_data.x, test_data.edge_index)
    test_proba  = F.softmax(test_out, dim=1)[:, 1].cpu().numpy()
    test_preds  = test_out.argmax(dim=1).cpu().numpy()
    test_labels = test_data.y.cpu().numpy()

gnn_results = {
    'Model':     'GraphSAGE',
    'Precision': round(precision_score(test_labels, test_preds, zero_division=0), 4),
    'Recall':    round(recall_score(test_labels, test_preds, zero_division=0), 4),
    'F1':        round(f1_score(test_labels, test_preds, zero_division=0), 4),
    'ROC-AUC':   round(roc_auc_score(test_labels, test_proba), 4),
    'PR-AUC':    round(average_precision_score(test_labels, test_proba), 4),
}

print('\n=== GRAPHSAGE TEST RESULTS ===')
for k, v in gnn_results.items():
    if k != 'Model':
        print(f'  {k:<12}: {v}')
print(f'\n{classification_report(test_labels, test_preds, target_names=["Legitimate","Fraud"])}')

with open('outputs/gnn_results.json', 'w') as f:
    json.dump(gnn_results, f, indent=2)

## 6. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(tr_losses, color='#2196F3')
ax1.set_title('GraphSAGE Training Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Cross-Entropy Loss')

ax2.plot(val_f1s, color='#4CAF50')
ax2.set_title('GraphSAGE Validation F1')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('F1 Score')

plt.tight_layout()
plt.savefig('outputs/gnn_training_curves.png', dpi=150)
plt.show()

## 7. Final Comparison Table

In [ ]:
baseline = pd.read_csv('outputs/baseline_results.csv')
lstm_r   = pd.read_json('outputs/lstm_results.json', typ='series').to_frame().T
gnn_r    = pd.DataFrame([gnn_results])

all_results = pd.concat([baseline, lstm_r, gnn_r], ignore_index=True)

print('\n' + '='*70)
print('  COMPLETE RESULTS COMPARISON')
print('='*70)
print(all_results.to_string(index=False))
all_results.to_csv('outputs/all_results.csv', index=False)

# Bar chart comparison
metrics = ['Precision', 'Recall', 'F1', 'PR-AUC']
fig, axes = plt.subplots(1, 4, figsize=(16, 5))

colors = ['#9E9E9E', '#607D8B', '#2196F3', '#4CAF50', '#FF9800']
for i, metric in enumerate(metrics):
    vals = all_results[metric].astype(float)
    bars = axes[i].bar(range(len(all_results)), vals, color=colors[:len(all_results)])
    axes[i].set_title(metric)
    axes[i].set_xticks(range(len(all_results)))
    axes[i].set_xticklabels(
        [m.replace(' ', '\n') for m in all_results['Model']],
        fontsize=7, rotation=0
    )
    axes[i].set_ylim(0, 1.1)
    for bar, val in zip(bars, vals):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                     f'{val:.3f}', ha='center', fontsize=7, fontweight='bold')

plt.suptitle('Model Comparison on PaySim Dataset', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/final_comparison.png', dpi=150)
plt.show()
print('\nAll outputs saved to outputs/')
print('Use outputs/all_results.csv and the PNG files in your paper.')